In [21]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch
from sklearn.metrics import accuracy_score
import evaluate


In [16]:
imdb = load_dataset("imdb")
test = imdb["test"]

In [19]:
sst2_model_path = "../results/finetuned_sst2_model"

tokenizer = AutoTokenizer.from_pretrained(sst2_model_path)
model = AutoModelForSequenceClassification.from_pretrained(sst2_model_path).to("cuda")
model.eval()

metric = evaluate.load("accuracy")

all_preds = []
all_labels = []

print("Running Zero-shot Evaluation on IMDb...")

for example in test:
    inputs = tokenizer(example["text"], truncation=True, padding="max_length", max_length=256, return_tensors="pt").to("cuda")
    labels = torch.tensor([example["label"]]).to("cuda")
    
    with torch.no_grad():
        outputs = model(**inputs)
        preds = outputs.logits.argmax(-1)
    
    all_preds.append(preds.item())
    all_labels.append(labels.item())

zero_shot_result = metric.compute(predictions=all_preds, references=all_labels)
print(f"Zero-shot IMDb Accuracy (SST-2 model without IMDb finetuning): {zero_shot_result['accuracy']:.4f}")

Running Zero-shot Evaluation on IMDb...
Zero-shot IMDb Accuracy (SST-2 model without IMDb finetuning): 0.8734


In [20]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

imdb = imdb.map(preprocess_function, batched=True)
imdb = imdb.remove_columns(["text"])
imdb.set_format("torch")

Map: 100%|██████████| 50000/50000 [00:13<00:00, 3799.70 examples/s]


In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir="./results/imdb_adapted_sst2",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=imdb["train"],
    eval_dataset=imdb["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\shiva\AppData\Local\Temp\ipykernel_7800\4175945036.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: shivamsinghml (shivamsingh-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.297300,0.313718,0.870800
2,0.201800,0.315288,0.882520
3,0.104700,0.446682,0.881720


TrainOutput(global_step=4689, training_loss=0.22465843421153409, metrics={'train_runtime': 370.1286, 'train_samples_per_second': 202.632, 'train_steps_per_second': 12.669, 'total_flos': 2483763724800000.0, 'train_loss': 0.22465843421153409, 'epoch': 3.0})

In [24]:
trainer.save_model("../results/imdb_adapted_sst2_final")
tokenizer.save_pretrained("../results/imdb_adapted_sst2_final")

('../results/imdb_adapted_sst2_final\\tokenizer_config.json',
 '../results/imdb_adapted_sst2_final\\special_tokens_map.json',
 '../results/imdb_adapted_sst2_final\\vocab.txt',
 '../results/imdb_adapted_sst2_final\\added_tokens.json',
 '../results/imdb_adapted_sst2_final\\tokenizer.json')

In [25]:
imdb_finetuned_model_path = "../results/imdb_adapted_sst2_final" 
model_imdb = AutoModelForSequenceClassification.from_pretrained(imdb_finetuned_model_path).to("cuda")
model_imdb.eval()

all_preds = []
all_labels = []

print("Running IMDb Fine-tuned Model Evaluation on IMDb...")

for example in test:
    inputs = tokenizer(example["text"], truncation=True, padding="max_length", max_length=256, return_tensors="pt").to("cuda")
    labels = torch.tensor([example["label"]]).to("cuda")
    
    with torch.no_grad():
        outputs = model_imdb(**inputs)
        preds = outputs.logits.argmax(-1)
    
    all_preds.append(preds.item())
    all_labels.append(labels.item())

imdb_finetuned_result = metric.compute(predictions=all_preds, references=all_labels)
print(f"IMDb Fine-tuned IMDb Accuracy: {imdb_finetuned_result['accuracy']:.4f}")


print("\n====== FINAL COMPARISON ======")
print(f"Zero-shot IMDb Accuracy: {zero_shot_result['accuracy']:.4f}")
print(f"IMDb Finetuned Accuracy:  {imdb_finetuned_result['accuracy']:.4f}")

if imdb_finetuned_result['accuracy'] > zero_shot_result['accuracy']:
    print("Domain adaptation helped improve IMDb performance!")
else:
    print("No improvement — IMDb domain very similar or SST-2 already generalized well.")

Running IMDb Fine-tuned Model Evaluation on IMDb...
IMDb Fine-tuned IMDb Accuracy: 0.9142

====== FINAL COMPARISON ======
Zero-shot IMDb Accuracy: 0.8734
IMDb Finetuned Accuracy:  0.9142
Domain adaptation helped improve IMDb performance!
